# Week 2: Text Classification

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week02-text-classification.html

**How to use:** run the cells from the top (Runtime → Run all, or Shift+Enter cell by cell). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need a key: add it with the 🔑 Secrets panel on the left, or set `os.environ["OPENAI_API_KEY"] = "..."` in a cell. Examples that use Ollama need a local Ollama server and will not work in Colab. For GPU examples choose Runtime → Change runtime type → T4 GPU.

In [ ]:
# Libraries this lesson uses that are not preinstalled in Colab
%pip install -q datasets gradio

In [ ]:
# Imports that some of the lesson's code fragments assume
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

## Mini-challenge: verify your setup

In [ ]:
import torch
import transformers
import datasets

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")

## Loading the dataset

In [ ]:
from datasets import load_dataset

emotions = load_dataset("dair-ai/emotion")
print(emotions)

## Examining dataset structure

In [ ]:
emotions["train"].features
emotions["train"].column_names
emotions["train"]["text"][:2]
emotions["train"]["label"][:2]

## Converting to pandas

In [ ]:
import pandas as pd

emotions.set_format(type="pandas")
df = emotions["train"][:]

def convert_label_int_2_string(row):
    return emotions["train"].features["label"].int2str(row)

df["label_name"] = df["label"].apply(convert_label_int_2_string)
df.head()

## Visualizing the data

In [ ]:
import matplotlib.pyplot as plt

df["label_name"].value_counts(ascending=True).plot.barh()
plt.title("Distribution of Emotions in Dataset")
plt.xlabel("Count")
plt.ylabel("Emotion")
plt.show()

## Mini-challenge: class percentages

In [ ]:
percentages = df["label_name"].value_counts(normalize=True) * 100
print(percentages)

## What is tokenization? (character-level example)

In [ ]:
text = "My name is kareem"
tokenized_text = list(text)
print(tokenized_text)

token2index = {ch: idx for idx, ch in enumerate(sorted(set(tokenized_text)))}
print(token2index)
# {' ': 0, 'M': 1, 'a': 2, 'e': 3, 'i': 4, 'k': 5, 'm': 6, 'n': 7, 'r': 8, 's': 9, 'y': 10}

## Using Hugging Face tokenizers

In [ ]:
from transformers import AutoTokenizer

model_ckp = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckp)

text = "My name is kareem"
encoded_text = tokenizer(text)
print(encoded_text)

tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
print(tokens)

print(tokenizer.convert_tokens_to_string(tokens))

## Tokenizer properties

In [ ]:
print(tokenizer.vocab_size)          # vocabulary size
print(tokenizer.model_input_names)   # inputs the model expects
print(tokenizer.model_max_length)    # maximum sequence length

## Tokenizing the entire dataset

In [ ]:
emotions.reset_format()   # undo the pandas format set earlier

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

emotion_encoded = emotions.map(tokenize, batched=True, batch_size=None)
print(emotion_encoded.column_names)

## Mini-challenge: tokenize a custom sentence

In [ ]:
sentence = "I am feeling both happy and sad at the same time."
encoded = tokenizer(sentence)
print(encoded.input_ids)
print(tokenizer.convert_ids_to_tokens(encoded.input_ids))

## Loading the pre-trained model

In [ ]:
from transformers import AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_chk = "distilbert-base-uncased"
model = AutoModel.from_pretrained(model_chk).to(device)   # base model, no classification head

## Feature extraction function

In [ ]:
def extract_hidden_states(batch):
    inputs = {k: v.to(device) for k, v in batch.items() if k in tokenizer.model_input_names}
    with torch.no_grad():
        last_hidden_state = model(**inputs).last_hidden_state
    return {"hidden_state": last_hidden_state[:, 0].cpu().numpy()}

## Processing the dataset

In [ ]:
emotion_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
emotions_hidden = emotion_encoded.map(extract_hidden_states, batched=True)

import numpy as np

X_train = np.array(emotions_hidden["train"]["hidden_state"])
X_valid = np.array(emotions_hidden["validation"]["hidden_state"])
y_train = np.array(emotions_hidden["train"]["label"])
y_valid = np.array(emotions_hidden["validation"]["label"])

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_valid.shape}")

## Mini-challenge: inspect the features

In [ ]:
print(f"Feature vector size: {X_train.shape[1]}")

labels = emotions["train"].features["label"].names
for i, emotion in enumerate(labels):
    mean_vec = X_train[y_train == i].mean(axis=0)
    print(f"{emotion}: {mean_vec[:5]} ...")

# Optional visualization
# !pip install umap-learn
# import umap
# reducer = umap.UMAP()
# X_2d = reducer.fit_transform(X_train)
# plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, cmap="viridis", s=2)
# plt.colorbar()
# plt.show()

## Dummy classifier

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_score = dummy_clf.score(X_valid, y_valid)
print(f"Dummy classifier accuracy: {dummy_score:.4f}")

## Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=3000)
lr_clf.fit(X_train, y_train)
lr_score = lr_clf.score(X_valid, y_valid)
print(f"Logistic regression accuracy: {lr_score:.4f}")
print(f"Improvement over dummy: {lr_score - dummy_score:.4f}")

## Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def plot_confusion_matrix(y_preds, y_true, labels):
    cm = confusion_matrix(y_true, y_preds, normalize="true")
    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title("Normalized confusion matrix")
    plt.show()

y_preds = lr_clf.predict(X_valid)
labels = emotions["train"].features["label"].names
plot_confusion_matrix(y_preds, y_valid, labels)

## Additional metrics

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_valid, y_preds, target_names=labels))

## Mini-challenge: random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
rf_score = rf_clf.score(X_valid, y_valid)
print(f"Random forest accuracy: {rf_score:.4f}  (logistic regression: {lr_score:.4f})")

rf_preds = rf_clf.predict(X_valid)
rf_report = classification_report(y_valid, rf_preds, target_names=labels)
print(rf_report)

## Loading a classification model

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = 6
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}
model = (AutoModelForSequenceClassification
         .from_pretrained(model_chk, num_labels=num_labels,
                          id2label=id2label, label2id=label2id)
         .to(device))

## Evaluation metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

## Setting up the Trainer

In [ ]:
from transformers import Trainer, TrainingArguments

batch_size = 64
logging_steps = len(emotion_encoded["train"]) // batch_size
model_name = f"{model_chk}-finetuned-emotion"

training_args = TrainingArguments(
    output_dir=model_name,
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy="epoch",
    disable_tqdm=False,
    logging_steps=logging_steps,
    push_to_hub=True,
    log_level="error",
)

trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=emotion_encoded["train"],
    eval_dataset=emotion_encoded["validation"],
    processing_class=tokenizer,
)

## Training the model

In [ ]:
trainer.train()

## Mini-challenge: tuning experiments

In [ ]:
# 1. Try a different learning rate
training_args.learning_rate = 5e-5

# 2. Try a different scheduler
training_args.lr_scheduler_type = "cosine"

# 3. More epochs with early stopping
from transformers import EarlyStoppingCallback
training_args.num_train_epochs = 5
training_args.save_strategy = "epoch"        # must match eval_strategy when loading the best model
training_args.load_best_model_at_end = True
training_args.metric_for_best_model = "f1"   # required by EarlyStoppingCallback
trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))

# Re-run trainer.train() after each change to see its effect

## Evaluating on the validation set

In [ ]:
validation_results = trainer.evaluate()
print(f"Validation accuracy: {validation_results['eval_accuracy']:.4f}")
print(f"Validation F1: {validation_results['eval_f1']:.4f}")
print(f"Improvement over logistic regression: {validation_results['eval_accuracy'] - lr_score:.4f}")

## Confusion matrix

In [ ]:
preds_output = trainer.predict(emotion_encoded["validation"])
y_preds = preds_output.predictions.argmax(-1)
y_valid = preds_output.label_ids
plot_confusion_matrix(y_preds, y_valid, labels)

## Detailed classification report

In [ ]:
print(classification_report(y_valid, y_preds, target_names=labels))

## Error analysis

In [ ]:
validation_texts = emotions["validation"]["text"]
validation_labels = emotions["validation"]["label"]

errors = []
for i, (pred, true) in enumerate(zip(y_preds, y_valid)):
    if pred != true:
        errors.append({
            "text": validation_texts[i],
            "true": labels[true],
            "predicted": labels[pred],
        })

for i, err in enumerate(errors[:5]):
    print(f"{i+1}. Text: {err['text']}")
    print(f"   True: {err['true']}, Predicted: {err['predicted']}\n")

## Mini-challenge: predict on custom text

In [ ]:
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][predicted_class].item()
    return {
        "emotion": labels[predicted_class],
        "confidence": confidence,
        "all_probs": {labels[i]: probs[0][i].item() for i in range(len(labels))},
    }

test_texts = [
    "I can't believe I aced my exam!",
    "I miss my family so much right now.",
    "That driver just cut me off! So rude!",
]
for text in test_texts:
    result = predict_emotion(text)
    print(f"Text: {text}")
    print(f"Predicted: {result['emotion']} (confidence: {result['confidence']:.4f})")
    print(f"All probabilities: {result['all_probs']}\n")

## Setting up your Hugging Face account

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Pushing the model to the Hub

In [ ]:
trainer.push_to_hub(commit_message="Training completed!")

## Creating a model card

In [ ]:
model_card = """
# Emotion Classification Model

This model classifies text into six emotions: sadness, joy, love, anger, fear, and surprise.

## Model Description
- Architecture: DistilBERT (distilbert-base-uncased)
- Fine-tuned on the DAIR.AI Emotion dataset
- Training accuracy: {train_acc:.4f}
- Validation accuracy: {val_acc:.4f}
- F1 score: {f1:.4f}

## Intended Usage
Sentiment analysis and emotion detection in English text.

## Limitations
- English only
- May struggle with sarcasm and ambiguous emotions
- May underperform on very short texts

## Training Procedure
- 2 epochs
- Learning rate: 2e-5
- Batch size: 64
""".format(
    train_acc=0.92,   # replace with your actual training accuracy
    val_acc=validation_results["eval_accuracy"],
    f1=validation_results["eval_f1"],
)

# from huggingface_hub import ModelCard
# ModelCard(model_card).push_to_hub("YOUR_USERNAME/distilbert-base-uncased-finetuned-emotion",
#                                   commit_message="Update model card")

## Using your model

In [ ]:
from transformers import pipeline

model_name = "YOUR_USERNAME/distilbert-base-uncased-finetuned-emotion"   # replace YOUR_USERNAME
classifier = pipeline("text-classification", model=model_name)

result = classifier("I'm so excited to see this working!")
print(result)   # e.g. [{'label': 'joy', 'score': 0.9874}]

## Mini-challenge (optional): Gradio demo

In [ ]:
!pip install gradio
import gradio as gr

def predict(text):
    results = classifier(text, top_k=None)   # scores for all six emotions
    return {r["label"]: r["score"] for r in results}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(placeholder="Enter text here..."),
    outputs=gr.Label(num_top_classes=6),
    title="Emotion Classifier",
    description="Classifies text into six emotions: sadness, joy, love, anger, fear, surprise.",
)
demo.launch()

## Run in your browser

**Sentiment classifier: TF-IDF features + logistic regression (scikit-learn)**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

train_texts = [
    "I love this course", "What a wonderful lecture", "The project was fun and useful",
    "Great explanation, very clear", "I enjoyed the lab today", "This tutorial is excellent",
    "I hate waiting for the model to train", "The lecture was boring", "This bug is terrible",
    "I am disappointed with the results", "The exam was awful", "Nothing works, very frustrating",
]
train_labels = ["pos"] * 6 + ["neg"] * 6
test_texts = ["The lab was clear and fun", "The results are terrible", "I love the project", "Boring and frustrating exam"]
test_labels = ["pos", "neg", "pos", "neg"]

vec = TfidfVectorizer()
X_train = vec.fit_transform(train_texts)
clf = LogisticRegression().fit(X_train, train_labels)

pred = clf.predict(vec.transform(test_texts))
for t, p in zip(test_texts, pred):
    print(f"{p:>3}  <- {t}")
print()
print(classification_report(test_labels, pred, zero_division=0))

**Precision, recall and F1 by hand vs scikit-learn**

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
y_pred = [1, 1, 1, 0, 1, 0, 0, 0, 0, 0]

tp = sum(t == 1 and p == 1 for t, p in zip(y_true, y_pred))
fp = sum(t == 0 and p == 1 for t, p in zip(y_true, y_pred))
fn = sum(t == 1 and p == 0 for t, p in zip(y_true, y_pred))
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"TP={tp} FP={fp} FN={fn}")
print(f"by hand:      precision={precision:.3f} recall={recall:.3f} F1={f1:.3f}")
print(f"scikit-learn: precision={precision_score(y_true, y_pred):.3f} "
      f"recall={recall_score(y_true, y_pred):.3f} F1={f1_score(y_true, y_pred):.3f}")

## Test code for the course environment

**Dataset + tokenizer sanity check (no training)**

In [ ]:
# 1. Dataset + tokenizer sanity check (no training)
from datasets import load_dataset
from transformers import AutoTokenizer

emotions = load_dataset("dair-ai/emotion")
labels = emotions["train"].features["label"].names
print(emotions)
print(labels)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
enc = tokenizer("I am feeling both happy and sad at the same time.")
print(enc.input_ids)
print(tokenizer.convert_ids_to_tokens(enc.input_ids))

**[CLS] feature extraction + logistic-regression baseline on a small subset**

In [ ]:
# 2. [CLS] feature extraction + logistic-regression baseline on a small subset
import numpy as np, torch
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device).eval()

def cls_features(texts, bs=32):
    feats = []
    for i in range(0, len(texts), bs):
        batch = tokenizer(texts[i:i+bs], padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            feats.append(model(**batch).last_hidden_state[:, 0].cpu().numpy())
    return np.vstack(feats)

train = emotions["train"].shuffle(seed=0).select(range(1000))
valid = emotions["validation"].select(range(300))
X_train, y_train = cls_features(train["text"]), np.array(train["label"])
X_valid, y_valid = cls_features(valid["text"]), np.array(valid["label"])
print("feature shape:", X_train.shape)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
lr = LogisticRegression(max_iter=3000).fit(X_train, y_train)
print(f"dummy: {dummy.score(X_valid, y_valid):.3f}   logistic regression: {lr.score(X_valid, y_valid):.3f}")

**Quick fine-tune on a subset (1 epoch) and evaluate**

In [ ]:
# 3. Quick fine-tune on a subset (1 epoch) and evaluate
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

small = emotions["train"].shuffle(seed=0).select(range(2000)).map(tokenize, batched=True)
small_val = emotions["validation"].select(range(500)).map(tokenize, batched=True)

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(pred.label_ids, preds),
            "f1": f1_score(pred.label_ids, preds, average="weighted")}

clf = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=6).to(device)
args = TrainingArguments(output_dir="distilbert-emotion-test", num_train_epochs=1, learning_rate=2e-5,
                         per_device_train_batch_size=32, per_device_eval_batch_size=32,
                         weight_decay=0.01, eval_strategy="epoch", report_to="none", log_level="error")
trainer = Trainer(model=clf, args=args, compute_metrics=compute_metrics,
                  train_dataset=small, eval_dataset=small_val, processing_class=tokenizer)
trainer.train()
print(trainer.evaluate())

**Predict emotion with confidence from the fine-tuned model**

In [ ]:
# 4. Predict emotion with confidence from the fine-tuned model
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(clf(**inputs).logits, dim=-1)[0]
    idx = int(probs.argmax())
    return labels[idx], float(probs[idx]), {l: round(float(p), 3) for l, p in zip(labels, probs)}

for t in ["I can't believe I aced my exam!",
          "I miss my family so much right now.",
          "That driver just cut me off! So rude!"]:
    print(t, "->", predict_emotion(t))